# FPR Experiment for Naive and Bonferroni across different nT
This notebook tests the false positive rate (FPR) of Naive and Bonferroni methods across varying target sample sizes (`nT`). It supports resumability, meaning if the execution is interrupted, it can pick up exactly where it left off by reading the saved `.npz` files.

In [21]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
REPO_ROOT = next((candidate for candidate in candidates if (candidate / "cort_si").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repo root containing the cort_si package.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from cort_si import (
    adaptive_source_selection,
    construct_Y,
    construct_Sigma,
    construct_folds,
    construct_test_statistic,
    gen_data,
    solve_cort_model,
)

RESULTS_DIR = REPO_ROOT / "run_experiment" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [22]:
p = 300
nS = 120
num_info_aux = 3
num_uninfo_aux = 2
K = num_info_aux + num_uninfo_aux
lambda_sel = 0.6
lambda0 = 0.7
T = 5
nT_list = [50, 75, 100]

num_runs = 1000
base_seed = 0
alpha = 0.05
true_beta_scale = 0.0
rho = 0.0
sigma_noise = 1.0
source_shift_sd = 0.3
covariate_shift = False
save_interval = 1

bonf_log10_family_size = p * np.log10(2.0)

print(f"num_runs = {num_runs}")
print(f"fixed alpha = {alpha}")
print(f"Null model true_beta scale = {true_beta_scale}")
print(f"K = {K}, p = {p}, nS = {nS}")
print(f"Testing nT values: {nT_list}")
print(f"Bonferroni family size uses 2^p with p = {p} (about 10^{bonf_log10_family_size:.1f})")

num_runs = 1000
fixed alpha = 0.05
Null model true_beta scale = 0.0
K = 5, p = 300, nS = 120
Testing nT values: [50, 75, 100]
Bonferroni family size uses 2^p with p = 300 (about 10^90.3)


In [23]:
def generate_null_instance(nT_val, iteration_seed):
    return gen_data.generate_data(
        p=p,
        nS=nS,
        nT=nT_val,
        K=K,
        true_beta=true_beta_scale,
        rho=rho,
        sigma_noise=sigma_noise,
        source_shift_sd=source_shift_sd,
        covariate_shift=covariate_shift,
        seed=iteration_seed,
        num_info_aux=num_info_aux,
        num_uninfo_aux=num_uninfo_aux
    )

def bonferroni_adjust_pvalue(naive_pvalue, num_features):
    if naive_pvalue is None or not np.isfinite(naive_pvalue):
        return np.nan
    if naive_pvalue <= 0.0:
        return 0.0

    log_adjusted = np.log(naive_pvalue) + (num_features * np.log(2.0))
    if log_adjusted >= 0.0:
        return 1.0
    return float(np.exp(log_adjusted))

def compute_random_selected_pvalues(X0, Y0, XS_list, YS_list, SigmaS_list, Sigma0, lambdak_list, feature_seed):
    folds = construct_folds(X0.shape[0], T=T, shuffle=False)
    I_obs = adaptive_source_selection(X0, Y0, XS_list, YS_list, folds, lambda_sel, verbose=False)
    _, beta0_hat, _, _ = solve_cort_model(
        X0,
        Y0,
        XS_list,
        YS_list,
        I_obs,
        lambda0,
        lambdak_list,
        verbose=False,
        label="Observed CoRT fit",
    )
    M_obs = [idx for idx, value in enumerate(beta0_hat) if value != 0]
    if not M_obs:
        return None

    feature_rng = random.Random(feature_seed)
    feature_idx = feature_rng.choice(M_obs)

    Y_obs = construct_Y(YS_list, Y0)
    Sigma = construct_Sigma(SigmaS_list, Sigma0)
    X0M = X0[:, M_obs]
    etaj, etajTy = construct_test_statistic(feature_idx, X0M, Y_obs, M_obs, Y0.shape[0], Y_obs.shape[0])

    variance = float(np.asarray(etaj.T @ Sigma @ etaj).item())
    if not np.isfinite(variance) or variance <= 0.0:
        return None

    z_score = float(etajTy / np.sqrt(variance))
    naive_pvalue = float(2.0 * stats.norm.sf(abs(z_score)))
    bonf_pvalue = bonferroni_adjust_pvalue(naive_pvalue, X0.shape[1])

    return {
        "feature_idx": int(feature_idx),
        "naive_pvalue": naive_pvalue,
        "bonf_pvalue": bonf_pvalue,
    }

def get_save_paths(p_val, nS_val, nT_val, K, lambda_sel, lambda0):
    naive_path = RESULTS_DIR / f"naive_fpr_p{p_val}_nS{nS_val}_nT{nT_val}_K{K}_lamsel{lambda_sel}_lam0{lambda0}.npz"
    bonf_path = RESULTS_DIR / f"bonf_fpr_p{p_val}_nS{nS_val}_nT{nT_val}_K{K}_lamsel{lambda_sel}_lam0{lambda0}.npz"
    return naive_path, bonf_path

def load_progress(filepath):
    if filepath.exists():
        data = np.load(filepath)
        if 'j_values' in data and 'null_pvalues' in data:
            return list(data['j_values']), list(data['null_pvalues'])
        elif 'null_pvalues' in data:
            return [], list(data['null_pvalues'])
    return [], []

def save_progress(filepath, j_values, null_pvalues, p_val, nS_val, nT_val, num_runs_val, alpha_val):
    np.savez(
        filepath,
        j_values=np.asarray(j_values, dtype=int),
        null_pvalues=np.asarray(null_pvalues, dtype=float),
        p=np.asarray([p_val], dtype=int),
        nS=np.asarray([nS_val], dtype=int),
        nT=np.asarray([nT_val], dtype=int),
        num_runs=np.asarray([num_runs_val], dtype=int),
        alpha=np.asarray([alpha_val], dtype=float),
    )

In [24]:
def run_resumable_fpr_experiment(nT_val, total_runs, seed_base, save_intvl=10):
    naive_path, bonf_path = get_save_paths(p, nS, nT_val, K, lambda_sel, lambda0)
    
    naive_j, naive_pvalues = load_progress(naive_path)
    bonf_j, bonf_pvalues = load_progress(bonf_path)
    
    start_iteration = len(naive_pvalues)
    
    if start_iteration >= total_runs:
        print(f"Experiment for nT={nT_val} already completed ({total_runs} runs).")
        return naive_j, naive_pvalues, bonf_j, bonf_pvalues
    
    print(f"Starting/Resuming experiment for nT={nT_val} from run {start_iteration}/{total_runs}...")
    cnt_naive = 0
    cnt_bonf = 0
    sum_valid = 0
    
    for iteration in range(start_iteration, total_runs):
        data_seed = seed_base + iteration
        XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, _ = generate_null_instance(nT_val, data_seed)
        lambdak_list = [lambda0] * len(XS_list)

        pvalue_record = compute_random_selected_pvalues(
            X0,
            Y0,
            XS_list,
            YS_list,
            SigmaS_list,
            Sigma0,
            lambdak_list,
            feature_seed=data_seed,
        )

        if pvalue_record is not None:
            naive_p_val = pvalue_record["naive_pvalue"]
            bonf_p_val = pvalue_record["bonf_pvalue"]
            feat_idx = pvalue_record["feature_idx"]
            
            if np.isfinite(naive_p_val):
                if naive_p_val <= alpha:
                    cnt_naive += 1
                if bonf_p_val <= alpha:
                    cnt_bonf += 1
                sum_valid += 1
                
                naive_pvalues.append(naive_p_val)
                naive_j.append(feat_idx)
                bonf_pvalues.append(bonf_p_val)
                bonf_j.append(feat_idx)

        if (iteration + 1) % save_intvl == 0 or (iteration + 1) == total_runs:
            save_progress(naive_path, naive_j, naive_pvalues, p, nS, nT_val, total_runs, alpha)
            save_progress(bonf_path, bonf_j, bonf_pvalues, p, nS, nT_val, total_runs, alpha)
            print(f"  [nT={nT_val}] Saved progress: {iteration + 1}/{total_runs} iterations processed.")
            if sum_valid > 0:
                print(f"Empirical FPR @ alpha={alpha}: Naive={(cnt_naive / sum_valid):.3f}, Bonf={(cnt_bonf / sum_valid):.3f} | valid runs: {sum_valid}")

    return naive_j, naive_pvalues, bonf_j, bonf_pvalues

In [25]:
for current_nT in nT_list:
    run_resumable_fpr_experiment(current_nT, num_runs, base_seed, save_intvl=save_interval)

Starting/Resuming experiment for nT=50 from run 864/1000...
  [nT=50] Saved progress: 865/1000 iterations processed.
Empirical FPR @ alpha=0.05: Naive=1.000, Bonf=0.000 | valid runs: 1
  [nT=50] Saved progress: 866/1000 iterations processed.
Empirical FPR @ alpha=0.05: Naive=1.000, Bonf=0.000 | valid runs: 1
  [nT=50] Saved progress: 867/1000 iterations processed.
Empirical FPR @ alpha=0.05: Naive=1.000, Bonf=0.000 | valid runs: 1
  [nT=50] Saved progress: 868/1000 iterations processed.
Empirical FPR @ alpha=0.05: Naive=1.000, Bonf=0.000 | valid runs: 1
  [nT=50] Saved progress: 869/1000 iterations processed.
Empirical FPR @ alpha=0.05: Naive=1.000, Bonf=0.000 | valid runs: 1
  [nT=50] Saved progress: 870/1000 iterations processed.
Empirical FPR @ alpha=0.05: Naive=0.500, Bonf=0.000 | valid runs: 2
  [nT=50] Saved progress: 871/1000 iterations processed.
Empirical FPR @ alpha=0.05: Naive=0.667, Bonf=0.000 | valid runs: 3
  [nT=50] Saved progress: 872/1000 iterations processed.
Empirica

KeyboardInterrupt: 

In [26]:
print("=== FPR Experiment Summary ===")
for current_nT in nT_list:
    naive_path, bonf_path = get_save_paths(p, nS, current_nT, K, lambda_sel, lambda0)
    _, naive_pvalues = load_progress(naive_path)
    _, bonf_pvalues = load_progress(bonf_path)
    
    if not naive_pvalues:
        print(f"nT={current_nT}: No data found.")
        continue
    
    naive_arr = np.asarray(naive_pvalues)
    bonf_arr = np.asarray(bonf_pvalues)
    
    naive_rejection_rate = (naive_arr <= alpha).mean()
    bonf_rejection_rate = (bonf_arr <= alpha).mean()
    print(f"nT={current_nT}: {len(naive_arr)} valid runs | Naive FPR @ alpha={alpha}: {naive_rejection_rate:.3f} | Bonf FPR @ alpha={alpha}: {bonf_rejection_rate:.3f}")

=== FPR Experiment Summary ===
nT=50: 953 valid runs | Naive FPR @ alpha=0.05: 0.407 | Bonf FPR @ alpha=0.05: 0.000
nT=75: 561 valid runs | Naive FPR @ alpha=0.05: 0.414 | Bonf FPR @ alpha=0.05: 0.000
nT=100: 301 valid runs | Naive FPR @ alpha=0.05: 0.329 | Bonf FPR @ alpha=0.05: 0.000
